# 🔱 Shiv AI Voice Cloning v3 (Fixed)
**Owner: Shri Ram Nag | PAISAWALA Channel**

### Features:
- 🎙️ Voice Clone | 🎛️ Voice Design | 🔤 Simple TTS | 📋 Instruct Mode
- ✅ Long script chunking — bich mein nahi rukta
- ✅ Modern dark UI

> ⚡ **Sabse pehle:** Runtime → Change runtime type → **T4 GPU** select karein!

In [ ]:
# ✅ STEP 1 — GPU Check
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    raise RuntimeError('❌ GPU nahi mila! Runtime → Change runtime type → T4 GPU!')

In [ ]:
# ✅ STEP 2 — Dependencies Install
!pip install -q gradio==4.44.0
!pip install -q huggingface_hub transformers==4.44.0 accelerate
!pip install -q scipy numpy
print('✅ Dependencies installed!')

In [ ]:
# ✅ STEP 3 — Model Download from HuggingFace
# Yahan model download hoga aur local path se load hoga
# Direct from_pretrained(repo_id) kaam nahi karta — isliye pehle download karo

import os
from huggingface_hub import snapshot_download

REPO_ID   = 'Shriramnag/Shiv-AI-Voice-Cloning'
LOCAL_DIR = '/content/Shiv-AI-Voice-Cloning'

if not os.path.exists(LOCAL_DIR) or len(os.listdir(LOCAL_DIR)) < 5:
    print(f'📥 Downloading model: {REPO_ID}')
    print('⏳ Please wait... (~3.27 GB total, 5-10 min depending on speed)')
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=LOCAL_DIR,
        local_dir_use_symlinks=False,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*']
    )
    print('✅ Download complete!')
else:
    print(f'✅ Model already at {LOCAL_DIR}')

# Downloaded files check karo
print('\n📂 Files in model folder:')
for f in sorted(os.listdir(LOCAL_DIR)):
    fpath = os.path.join(LOCAL_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {f:45s} {size_mb:.1f} MB')
    else:
        print(f'  {f}/ (folder)')

In [ ]:
# ✅ STEP 4 — Model Load (Local Path se)
import sys
import torch
import numpy as np

LOCAL_DIR = '/content/Shiv-AI-Voice-Cloning'

# Local path ko Python path mein add karo
sys.path.insert(0, LOCAL_DIR)

# Ab omnivoice import karo
try:
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig
    from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name
    print('✅ OmniVoice imported!')
except ImportError as e:
    print(f'⚠️ Import error: {e}')
    print('Trying alternative import path...')
    sys.path.insert(0, f'{LOCAL_DIR}/OmniVoice')
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig
    from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

try:
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except ImportError:
    WHISPER_LANGUAGE_CODE = None
    print('Note: subtitle module not found (ok to continue)')

# Model load karo LOCAL path se (repo_id se nahi)
print(f'🔱 Loading Shiv AI model from {LOCAL_DIR}...')
model = OmniVoice.from_pretrained(
    LOCAL_DIR,          # ← LOCAL path, HuggingFace repo id nahi
    device_map='cuda',
    dtype=torch.float16,
    load_asr=False,
)
SR = model.sampling_rate
print(f'✅ Shiv AI Model loaded! Sampling Rate: {SR} Hz')

In [ ]:
# ✅ STEP 5 — Helper Functions (Chunking + All 4 tabs)
import re, os, gradio as gr
os.makedirs('/content/Shiv_Audio', exist_ok=True)

LANG_CHOICES = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)
EVENT_TAGS   = ['[laughter]','[sigh]','[confirmation-en]','[question-en]','[surprise-wa]','[dissatisfaction-hnn]']
INSTRUCT_EX  = [
    'Speak slowly and clearly with a calm, deep voice',
    'Speak with excitement and high energy',
    'Speak softly like a bedtime story narrator',
    'Speak like a professional news anchor, formal and clear',
    'Speak in a sad, emotional tone with pauses',
    'Fast and enthusiastic like a radio jockey',
    'धीरे, शांत और गहरी आवाज़ में बोलें',
    'जोश और उत्साह के साथ तेज़ आवाज़ में बोलें',
]
INSERT_TAG_JS = """
(tag_val, current_text) => {
    const ta = document.querySelector('.shiv-tb textarea');
    if (!ta) return current_text + ' ' + tag_val;
    const s = ta.selectionStart, e = ta.selectionEnd;
    return current_text.slice(0,s) + ' ' + tag_val + ' ' + current_text.slice(e);
}
"""

# ── Chunking ──────────────────────────────────────
def split_chunks(text, max_ch=120):
    lines = re.split(r'(?:\u2026\n?|\u0964\n|\n)', text)
    lines = [l.strip() for l in lines if l.strip()]
    chunks, cur = [], ''
    for line in lines:
        if len(line) > max_ch:
            if cur: chunks.append(cur); cur = ''
            for sent in re.split(r'(?<=[\u0964.!?])\s+', line):
                if len(cur)+len(sent)+1 <= max_ch: cur = (cur+' '+sent).strip()
                else:
                    if cur: chunks.append(cur)
                    cur = sent.strip()
        else:
            if len(cur)+len(line)+1 <= max_ch: cur = (cur+' '+line).strip()
            else:
                if cur: chunks.append(cur)
                cur = line.strip()
    if cur: chunks.append(cur)
    return [c for c in chunks if c.strip()]

def join_chunks(audios, silence_ms=0):
    if silence_ms > 0:
        sil = np.zeros(int(SR*silence_ms/1000), dtype=np.float32)
        parts = []
        for i,a in enumerate(audios):
            parts.append(a)
            if i < len(audios)-1: parts.append(sil)
        return np.concatenate(parts)
    return np.concatenate(audios)

def make_cfg(steps=32, gs=2.0, speed=1.0, pitch=0, energy=1.0):
    try:
        return OmniVoiceGenerationConfig(
            num_step=steps, guidance_scale=gs, denoise=True,
            preprocess_prompt=True, postprocess_output=True,
            speed=speed, pitch=pitch, energy=energy)
    except TypeError:
        return OmniVoiceGenerationConfig(
            num_step=steps, guidance_scale=gs, denoise=True,
            preprocess_prompt=True, postprocess_output=True)

def run_chunk(text, lang, cfg, vcp=None, instruct=None):
    kw = dict(text=text, language=lang if lang!='Auto' else None, generation_config=cfg)
    if vcp:     kw['voice_clone_prompt'] = vcp
    if instruct: kw['instruct'] = instruct
    return model.generate(**kw)[0]

def to_wav(a): return (SR, (a*32767).astype(np.int16))

# ── 4 Main Functions ──────────────────────────────
def fn_clone(text, lang, ref, ref_text):
    if not text  or not text.strip(): return None, '⚠️ Text likhein'
    if not ref:                        return None, '⚠️ Reference audio upload karein'
    try:
        vcp = model.create_voice_clone_prompt(ref_audio=ref, ref_text=ref_text.strip() or None)
        cfg = make_cfg()
        chunks = split_chunks(text)
        print(f'[Clone] {len(chunks)} chunks')
        audio = join_chunks([run_chunk(c,lang,cfg,vcp=vcp) for c in chunks], silence_ms=0)
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_design(text, lang, speed, pitch, energy, pause_ms, style):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        cfg  = make_cfg(gs=2.5, speed=speed, pitch=pitch, energy=energy)
        inst = style.strip() or None
        chunks = split_chunks(text)
        print(f'[Design] {len(chunks)} chunks')
        audio = join_chunks([run_chunk(c,lang,cfg,instruct=inst) for c in chunks], silence_ms=int(pause_ms))
        return to_wav(audio), f'✅ Done! speed={speed} pitch={pitch} energy={energy} | {len(audio)/SR:.1f}s'
    except Exception as e:
        try:
            cfg = make_cfg(gs=2.5)
            chunks = split_chunks(text)
            audio = join_chunks([run_chunk(c,lang,cfg,instruct=style.strip() or None) for c in chunks])
            return to_wav(audio), f'✅ Done (basic)! {len(audio)/SR:.1f}s'
        except Exception as e2: return None, f'❌ {e2}'

def fn_tts(text, lang, steps, gs):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        cfg = make_cfg(int(steps), float(gs))
        chunks = split_chunks(text)
        print(f'[TTS] {len(chunks)} chunks')
        audio = join_chunks([run_chunk(c,lang,cfg) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_instruct(text, lang, prompt):
    if not text   or not text.strip():   return None, '⚠️ Text likhein'
    if not prompt or not prompt.strip(): return None, '⚠️ Instruction likhein'
    try:
        cfg = make_cfg(gs=3.0)
        chunks = split_chunks(text)
        print(f'[Instruct] {len(chunks)} chunks')
        audio = join_chunks([run_chunk(c,lang,cfg,instruct=prompt.strip()) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

print('✅ All functions ready!')

In [ ]:
# ✅ STEP 6 — Launch Shiv AI UI
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@600;800&family=DM+Sans:wght@300;400;500&display=swap');
:root{--saffron:#FF6B00;--gold:#FFB347;--dark:#0A0A0C;--surf:#111115;--border:rgba(255,107,0,0.25);--text:#EDEAE2;--muted:#7A7870;--r:12px;}
body,.gradio-container{background:var(--dark)!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;max-width:100%!important;padding:0!important;}
footer{display:none!important;}
.shiv-header{background:linear-gradient(160deg,#0f0800,#1a0d00 40%,#0A0A0C);border-bottom:1px solid var(--border);text-align:center;padding:32px 20px 24px;}
.shiv-header h1{font-family:'Syne',sans-serif!important;font-size:clamp(1.8em,4vw,2.8em);font-weight:800;background:linear-gradient(90deg,#FF6B00,#FFD580,#FF6B00);background-size:200%;-webkit-background-clip:text;-webkit-text-fill-color:transparent;animation:shimmer 3s linear infinite;margin-bottom:8px;}
@keyframes shimmer{to{background-position:200% center;}}
.shiv-header p{color:var(--muted);font-size:0.88em;margin:3px 0;}
.shiv-header b{color:var(--gold);}
.tab-nav{background:var(--surf)!important;border-bottom:1px solid var(--border)!important;padding:0 16px!important;}
.tab-nav button{font-family:'DM Sans',sans-serif!important;font-size:0.85em!important;color:var(--muted)!important;background:transparent!important;border:none!important;border-bottom:2px solid transparent!important;padding:13px 16px!important;border-radius:0!important;transition:all .2s!important;}
.tab-nav button.selected,.tab-nav button:hover{color:var(--saffron)!important;border-bottom-color:var(--saffron)!important;}
.tabitem{background:var(--dark)!important;padding:20px!important;}
.sec-title{font-family:'Syne',sans-serif!important;font-size:0.82em!important;font-weight:700!important;letter-spacing:0.12em!important;text-transform:uppercase!important;color:var(--saffron)!important;margin:0 0 12px!important;}
label span,.label-wrap span{font-size:0.75em!important;font-weight:500!important;letter-spacing:0.08em!important;text-transform:uppercase!important;color:var(--gold)!important;}
.shiv-tb textarea,textarea{background:#1C1C22!important;color:#EDEAE2!important;border:1.5px solid rgba(255,107,0,0.2)!important;border-radius:var(--r)!important;font-family:'DM Sans',sans-serif!important;font-size:0.95em!important;padding:12px 14px!important;caret-color:var(--saffron)!important;transition:border-color .25s,box-shadow .25s!important;}
textarea:focus{border-color:var(--saffron)!important;box-shadow:0 0 0 3px rgba(255,107,0,0.15)!important;outline:none!important;}
textarea::placeholder{color:#555550!important;font-style:italic!important;}
.wrap .wrap-inner,select{background:#1C1C22!important;border:1.5px solid rgba(255,107,0,0.2)!important;border-radius:var(--r)!important;color:var(--text)!important;}
input[type=range]{accent-color:var(--saffron)!important;}
div[data-testid='audio']{background:#18181E!important;border:1px solid var(--border)!important;border-radius:var(--r)!important;padding:10px!important;}
div[data-testid='audio'] button{background:var(--saffron)!important;border-radius:50%!important;width:36px!important;height:36px!important;border:none!important;color:#fff!important;transition:transform .15s,box-shadow .15s!important;}
div[data-testid='audio'] button:hover{transform:scale(1.1)!important;box-shadow:0 0 16px rgba(255,107,0,0.5)!important;}
.status-box textarea{background:#111118!important;border:1px solid rgba(255,255,255,0.06)!important;color:var(--gold)!important;font-size:0.82em!important;border-radius:8px!important;}
.btn-gen{background:linear-gradient(135deg,#FF6B00,#CC4400)!important;color:#fff!important;border:none!important;border-radius:var(--r)!important;font-family:'Syne',sans-serif!important;font-weight:700!important;font-size:1em!important;padding:14px 24px!important;width:100%!important;cursor:pointer!important;box-shadow:0 4px 24px rgba(255,107,0,0.4)!important;transition:transform .2s,box-shadow .2s!important;}
.btn-gen:hover{transform:translateY(-3px)!important;box-shadow:0 8px 32px rgba(255,107,0,0.6)!important;}
.btn-gen:active{transform:translateY(-1px)!important;}
.tag-btn{background:rgba(255,107,0,0.08)!important;border:1px solid rgba(255,107,0,0.25)!important;color:var(--gold)!important;border-radius:20px!important;font-size:0.72em!important;padding:5px 10px!important;transition:all .2s!important;}
.tag-btn:hover{background:rgba(255,107,0,0.22)!important;transform:scale(1.05)!important;}
.preset-btn{background:rgba(255,255,255,0.04)!important;border:1px solid rgba(255,255,255,0.08)!important;color:var(--muted)!important;border-radius:8px!important;font-size:0.82em!important;padding:9px 12px!important;flex:1!important;transition:all .2s!important;}
.preset-btn:hover{background:rgba(255,107,0,0.14)!important;border-color:rgba(255,107,0,0.5)!important;color:var(--gold)!important;transform:translateY(-2px)!important;}
.ex-btn{background:transparent!important;border:1px solid rgba(255,255,255,0.07)!important;color:var(--muted)!important;border-radius:7px!important;font-size:0.8em!important;padding:7px 12px!important;text-align:left!important;width:100%!important;transition:all .18s!important;margin-bottom:5px!important;}
.ex-btn:hover{background:rgba(255,107,0,0.10)!important;border-color:rgba(255,107,0,0.4)!important;color:var(--gold)!important;padding-left:16px!important;}
.info-card{background:rgba(255,107,0,0.05);border:1px solid rgba(255,107,0,0.15);border-radius:var(--r);padding:14px 16px;margin-top:12px;font-size:0.84em;color:var(--muted);line-height:1.8;}
.info-card strong{color:var(--gold);}
.info-card code{background:rgba(255,107,0,0.12);color:var(--saffron);padding:1px 5px;border-radius:4px;}
.divider{border:none!important;border-top:1px solid rgba(255,107,0,0.12)!important;margin:16px 0!important;}
.shiv-footer{text-align:center;padding:18px;color:var(--muted);font-size:0.8em;border-top:1px solid var(--border);background:var(--surf);}
.shiv-footer span{color:var(--saffron);}
"""

with gr.Blocks(css=CSS, title='🔱 Shiv AI Voice Cloning') as demo:

    gr.HTML("""
    <div class='shiv-header'>
      <h1>🔱 Shiv AI Voice Cloning</h1>
      <p>Advanced Multilingual Neural Speech Engine &nbsp;·&nbsp; 646 Languages</p>
      <p><b>Shri Ram Nag</b> &nbsp;·&nbsp; PAISAWALA 🎬 &nbsp;·&nbsp; v3 Fixed</p>
    </div>""")

    with gr.Tabs(elem_classes='tab-nav'):

        # TAB 1 — Voice Clone
        with gr.TabItem('🎙️ Voice Clone'):
            with gr.Row():
                with gr.Column(scale=11):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    vc_text = gr.Textbox(lines=9, elem_classes='shiv-tb', label='', show_label=False,
                                          placeholder='पूरी script paste करें — लंबी script भी चलेगी…')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b.click(fn=None, inputs=[b,vc_text], outputs=vc_text, js=INSERT_TAG_JS)
                    with gr.Row():
                        vc_lang     = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language', scale=1)
                        vc_ref_text = gr.Textbox(label='📄 Reference Transcript (optional)', lines=1, scale=2)
                    vc_ref = gr.Audio(label='🎤 Reference Audio (5–30 sec, saaf awaaz)', type='filepath')
                    vc_btn = gr.Button('🔱  Clone Voice & Generate', elem_classes='btn-gen', variant='primary')
                with gr.Column(scale=9):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    vc_out    = gr.Audio(type='numpy', label='Generated Audio', show_download_button=True)
                    vc_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box', lines=2)
                    gr.HTML('<div class="info-card"><strong>✅ Hakla issue fixed!</strong><br>Zero extra silence = no stutter<br><br><strong>💡 Tips:</strong><br>· 5–30 sec saaf reference audio<br>· <code>…</code> = natural pause</div>')
            vc_btn.click(fn_clone, [vc_text,vc_lang,vc_ref,vc_ref_text], [vc_out,vc_status])

        # TAB 2 — Voice Design
        with gr.TabItem('🎛️ Voice Design'):
            with gr.Row():
                with gr.Column(scale=11):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    vd_text = gr.Textbox(lines=6, elem_classes='shiv-tb', label='', show_label=False,
                                          placeholder='यहाँ text paste करें…')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b2 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b2.click(fn=None, inputs=[b2,vd_text], outputs=vd_text, js=INSERT_TAG_JS)
                    vd_lang   = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language')
                    gr.HTML('<hr class="divider"><div class="sec-title">🎚️ Voice Controls</div>')
                    vd_speed  = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label='⚡ Speed  — 0.5=Slow | 1.0=Normal | 2.0=Fast')
                    vd_pitch  = gr.Slider(-12, 12,  value=0,   step=1,    label='🎵 Pitch  — -12=Deep | 0=Normal | +12=High')
                    vd_energy = gr.Slider(0.3, 2.0, value=1.0, step=0.05, label='💪 Energy — 0.3=Whisper | 1.0=Normal | 2.0=Loud')
                    vd_pause  = gr.Slider(0, 400, value=0, step=50,       label='⏸️ Extra Pause (ms) — 0=recommended')
                    vd_style  = gr.Textbox(label='✍️ Style Instruction (optional)', lines=2, elem_classes='shiv-tb',
                                            placeholder='e.g. speak like a calm narrator…')
                    gr.HTML('<hr class="divider"><div class="sec-title">⚡ Quick Presets</div>')
                    with gr.Row():
                        pc=gr.Button('😌 Calm',elem_classes='preset-btn')
                        pe=gr.Button('🔥 Excited',elem_classes='preset-btn')
                        pn=gr.Button('📺 News',elem_classes='preset-btn')
                        ps=gr.Button('📖 Story',elem_classes='preset-btn')
                    vd_btn = gr.Button('🎛️  Design & Generate', elem_classes='btn-gen', variant='primary')
                with gr.Column(scale=9):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    vd_out    = gr.Audio(type='numpy', label='Voice Design Output', show_download_button=True)
                    vd_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box', lines=2)
                    gr.HTML('<div class="info-card"><strong>🎚️ Guide:</strong><br>Speed ↓=slow ↑=fast<br>Pitch ↓=deep ↑=high<br>Energy ↓=soft ↑=loud<br>Pause=silence btwn chunks (0=best)</div>')
            pc.click(fn=lambda:(0.8,-2,0.7,0,'speak calmly and peacefully'),           outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pe.click(fn=lambda:(1.3,3,1.5,0,'speak with excitement and high energy'),  outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pn.click(fn=lambda:(1.0,0,1.1,0,'speak like a professional news anchor'),  outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            ps.click(fn=lambda:(0.85,-1,0.8,100,'speak like a warm storyteller'),      outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            vd_btn.click(fn_design,[vd_text,vd_lang,vd_speed,vd_pitch,vd_energy,vd_pause,vd_style],[vd_out,vd_status])

        # TAB 3 — Simple TTS
        with gr.TabItem('🔤 Simple TTS'):
            with gr.Row():
                with gr.Column(scale=11):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    tts_text = gr.Textbox(lines=9, elem_classes='shiv-tb', label='', show_label=False,
                                           placeholder='यहाँ text paste करें — reference audio ki zaroorat nahi…')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b3=gr.Button(tag,elem_classes='tag-btn',size='sm')
                            b3.click(fn=None,inputs=[b3,tts_text],outputs=tts_text,js=INSERT_TAG_JS)
                    tts_lang = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language')
                    gr.HTML('<hr class="divider"><div class="sec-title">🎯 Quality Settings</div>')
                    with gr.Row():
                        tts_steps=gr.Slider(10,64,value=40,step=2,label='🔢 Steps — ↑ quality ↓ speed',scale=1)
                        tts_gs   =gr.Slider(1.0,5.0,value=3.0,step=0.5,label='🎯 Guidance — ↑ realistic',scale=1)
                    tts_btn = gr.Button('🔤  Generate HD Audio', elem_classes='btn-gen', variant='primary')
                with gr.Column(scale=9):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    tts_out    = gr.Audio(type='numpy', label='TTS Output', show_download_button=True)
                    tts_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box', lines=2)
                    gr.HTML('<div class="info-card"><strong>🎯 HD Tips:</strong><br>Steps=40, Guidance=3.0 → best balance<br>Steps=60, Guidance=4.0 → max quality<br>Steps=20, Guidance=2.0 → fast draft<br><br><code>…</code> = pause &nbsp;|&nbsp; <code>[laughter]</code> = hansi</div>')
            tts_btn.click(fn_tts,[tts_text,tts_lang,tts_steps,tts_gs],[tts_out,tts_status])

        # TAB 4 — Instruct Mode
        with gr.TabItem('📋 Instruct Mode'):
            with gr.Row():
                with gr.Column(scale=11):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    inst_text = gr.Textbox(lines=7, elem_classes='shiv-tb', label='', show_label=False,
                                            placeholder='यहाँ text paste करें…')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b4=gr.Button(tag,elem_classes='tag-btn',size='sm')
                            b4.click(fn=None,inputs=[b4,inst_text],outputs=inst_text,js=INSERT_TAG_JS)
                    inst_lang   = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language')
                    inst_prompt = gr.Textbox(label='📋 Style Instruction', lines=3, elem_classes='shiv-tb',
                                              placeholder='Speak slowly and clearly in a calm, deep voice…')
                    gr.HTML('<hr class="divider"><div class="sec-title">💡 Example Instructions</div>')
                    for ex in INSTRUCT_EX:
                        eb=gr.Button(ex, elem_classes='ex-btn', size='sm')
                        eb.click(fn=lambda x=ex: x, outputs=inst_prompt)
                    inst_btn = gr.Button('📋  Generate with Instruction', elem_classes='btn-gen', variant='primary')
                with gr.Column(scale=9):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    inst_out    = gr.Audio(type='numpy', label='Instruct Output', show_download_button=True)
                    inst_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box', lines=2)
                    gr.HTML('<div class="info-card"><strong>📋 Tips:</strong><br>English instructions best kaam karte hain<br><br><em>"Speak like a Bollywood trailer narrator"</em><br><em>"Old wise grandfather telling a story"</em><br><em>"Soft and emotional like a love letter"</em></div>')
            inst_btn.click(fn_instruct,[inst_text,inst_lang,inst_prompt],[inst_out,inst_status])

    gr.HTML("<div class='shiv-footer'>© 2026 <span>🔱 Shiv AI Voice Cloning</span> · <span>Shri Ram Nag</span> · PAISAWALA 🎬</div>")

demo.launch(share=True, debug=False)
print('🔱 Shiv AI launched! Upar wala public URL copy karein.')